In [1]:
import numpy as np

**Checking the SetCoverPy:**
To see how it works on its own test data

In [2]:
pip install SetCoverPy

  Preparing metadata (setup.py) ... done
  Created wheel for SetCoverPy: filename=SetCoverPy-0.9.1-py3-none-any.whl size=9858 sha256=abae905705d6164ae2d8732b6a06cd016a38633429eede4e85f5661df6774477
  Stored in directory: /root/.cache/pip/wheels/d8/90/6a/b882ac0211482e459ecb9dde12d096f76ed6c1225b9564def1
Successfully built SetCoverPy


In [3]:
import os
import urllib.request
import tarfile

# Set the URL for the test data archive
url = "https://s3.us-east-2.amazonaws.com/setcoverproblem/BeasleyOR/scp_BeasleyOR_numpyformat.tar.gz"

# Set the destination directory for extraction
destination_dir = "BeasleyOR"

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Download the test data archive
tar_gz_file = os.path.join(destination_dir, "scp_BeasleyOR_numpyformat.tar.gz")
urllib.request.urlretrieve(url, tar_gz_file)

# Extract the contents of the archive
with tarfile.open(tar_gz_file, "r:gz") as tar:
    tar.extractall(destination_dir)

In [4]:
from SetCoverPy import setcover

# Load the test data
# Update the paths based on the correct directory structure
a_matrix_path = "/content/BeasleyOR/scp_BeasleyOR_numpyformat/scpb4_matrix.npy"
cost_path = "/content/BeasleyOR/scp_BeasleyOR_numpyformat/scpb4_cost.npy"

# Load the test data
a_matrix = np.load(a_matrix_path)
cost = np.load(cost_path)

# Instantiate SetCover object
g = setcover.SetCover(a_matrix, cost)

# Run the solver
solution, time_used = g.SolveSCP()

This Best solution: UB=80.0, LB=70.39749213779288, UB1=86.0, LB1=70.64807279849141
Current Best Solution: UB=80, LB=70.39749213779288, change=12.003134827758899% @ niters=0
This Best solution: UB=80.0, LB=70.38666574630386, UB1=85.0, LB1=70.47026899552596
Current Best Solution: UB=79, LB=70.1577281140902, change=11.192749222670626% @ niters=5
This Best solution: UB=89.0, LB=-117.35929287940807, UB1=89.0, LB1=-117.35929287940807
Current Best Solution: UB=79, LB=70.70880310635273, change=10.495185941325655% @ niters=10
This Best solution: UB=81.0, LB=70.65583004288109, UB1=83.0, LB1=70.71249025393121
Current Best Solution: UB=79, LB=70.45089428290085, change=10.821652806454624% @ niters=15
Iteration in re-initialization reaches maximum number = 20
Current Best Solution: UB=79, LB=70.50053103039265, change=10.758821480515632% @ niters=20
Final Best solution: 79
Took 2.329 minutes to reach current solution.


In [5]:
# Print the loaded test data
print("Binary Relationship Matrix (a_matrix):")
print(a_matrix)

print("\nCost of Columns (cost):")
print(cost)

Binary Relationship Matrix (a_matrix):
[[ True False False ... False False False]
 [False False False ... False False False]
 [ True False False ... False False False]
 ...
 [ True False False ... False False False]
 [False False False ... False False False]
 [ True False False ... False False False]]

Cost of Columns (cost):
[  1   1   1 ... 100 100 100]


In [6]:
# Print the solution and total cost
print("Solution (Selected Columns):", solution)
print("Total Cost of the Set Cover:", g.total_cost)

Solution (Selected Columns): 79
Total Cost of the Set Cover: 79


In [7]:
print("Solution (g.s):", g.s)
print("Total Cost of the Set Cover:", g.total_cost)

Solution (g.s): [ True  True  True ... False False False]
Total Cost of the Set Cover: 79


**New Way to build Matrix**

In [3]:
from SetCoverPy import setcover

In [4]:
def generate_binary_matrices2(num_matrices, max_universe_size):
    binary_matrices = []

    for _ in range(num_matrices):
        # Generate a random number of rows and columns for each matrix
        num_rows = np.random.randint(3, max_universe_size + 1) #min universe size is 3
        num_cols = np.random.randint(1, max_universe_size + 1)

         # Create a random binary matrix with at least one True value in each row
        binary_matrix = np.zeros((num_rows, num_cols), dtype=bool)
        for i in range(num_rows):
            # Ensure at least one True value in each row
            binary_matrix[i, np.random.randint(0, num_cols)] = True

            # Ensure at least one True value in each column
        for j in range(num_cols):
            binary_matrix[np.random.randint(0, num_rows), j] = True

        binary_matrices.append(binary_matrix)

    return binary_matrices

In [5]:
# define parameters
num_matrices = 10
max_universe_size = 6

# Generate binary matrices
generated_matrices = generate_binary_matrices2(num_matrices, max_universe_size)

In [6]:
# Print the generated matrices
for i, matrix in enumerate(generated_matrices):
    print(f"Generated Binary Matrix {i + 1}:")
    print(matrix)
    print()

Generated Binary Matrix 1:
[[False  True False False False]
 [ True False False False False]
 [False False  True  True False]
 [False False False  True False]
 [ True  True False  True  True]
 [False False  True False False]]

Generated Binary Matrix 2:
[[ True False False  True False]
 [False False  True False False]
 [ True  True False False  True]]

Generated Binary Matrix 3:
[[False False  True False]
 [False  True False  True]
 [False False  True  True]
 [ True False False  True]
 [False False False  True]
 [False  True False False]]

Generated Binary Matrix 4:
[[ True  True False False False]
 [False False  True False False]
 [False False  True  True  True]]

Generated Binary Matrix 5:
[[False False False  True  True]
 [ True  True False  True False]
 [False False  True False False]
 [ True False False False False]]

Generated Binary Matrix 6:
[[False False  True False]
 [False False False  True]
 [ True  True False False]
 [ True False  True  True]
 [False False False  True]
 [F

In [7]:
# Function to convert a binary matrix to SetCoverPy input format
def convert_to_setcover_input(binary_matrix):
    return binary_matrix, np.zeros(binary_matrix.shape[1], dtype=float)

# List to store SetCoverPy solutions
setcover_solutions = []

# Iterate over the matrices and find set cover solutions
for i, generated_matrix in enumerate(generated_matrices):
    # Convert binary matrix to SetCoverPy input format
    setcover_input = convert_to_setcover_input(generated_matrix)

    # Instantiate SetCover object
    g = setcover.SetCover(*setcover_input)

    # Run the solver
    solution, time_used = g.SolveSCP()

    # Save the solution to the list
    setcover_solutions.append(g.s)

    # Print the solution and total cost
    print(f"Set Cover Solution for Matrix {i + 1}:")
    print("Solution (g.s):", g.s)
    print("Total Cost of the Set Cover:", g.total_cost)
    print()

# Now 'setcover_solutions' contains the binary vectors of solutions for each matrix

/usr/local/lib/python3.10/dist-packages/SetCoverPy/setcover.py:364: RuntimeWarning: invalid value encountered in scalar divide
  u_temp = u_sequence[:,i]+self._stepsize*(UB - Lu_sequence[i])/s_u_norm*s_u
/usr/local/lib/python3.10/dist-packages/SetCoverPy/setcover.py:484: RuntimeWarning: divide by zero encountered in scalar divide
  GAP = (UB-LB)/np.fabs(UB)


This Best solution: UB=0.0, LB=1e-05, UB1=0.0, LB1=1e-05
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=0
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=1
Final Best solution: 0.0
Took 0.001 minutes to reach current solution.
Set Cover Solution for Matrix 1:
Solution (g.s): [ True  True  True  True False]
Total Cost of the Set Cover: 0.0

This Best solution: UB=0.0, LB=-0.0159053125144683, UB1=0.0, LB1=-0.0159053125144683
Current Best Solution: UB=0.0, LB=-0.0159053125144683, change=inf% @ niters=0
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=2
Final Best solution: 0.0
Took 0.003 minutes to reach current solution.
Set Cover Solution for Matrix 2:
Solution (g.s): [ True False  True False False]
Total Cost of the Set Cover: 0.0

This Best solution: UB=0.0, LB=1e-05, UB1=0.0, LB1=1e-05
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=0
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=1
Final Best solution: 0.0
T

In [8]:
# Print the setcover_solutions list separately
for i, solution in enumerate(setcover_solutions):
    print(f"Set Cover Solution for Matrix {i + 1}:")
    print("Solution (g.s):", solution)
    print()

Set Cover Solution for Matrix 1:
Solution (g.s): [ True  True  True  True False]

Set Cover Solution for Matrix 2:
Solution (g.s): [ True False  True False False]

Set Cover Solution for Matrix 3:
Solution (g.s): [False  True  True  True]

Set Cover Solution for Matrix 4:
Solution (g.s): [ True False  True False False]

Set Cover Solution for Matrix 5:
Solution (g.s): [ True  True  True  True False]

Set Cover Solution for Matrix 6:
Solution (g.s): [False  True  True  True]

Set Cover Solution for Matrix 7:
Solution (g.s): [ True False  True False]

Set Cover Solution for Matrix 8:
Solution (g.s): [False  True False  True]

Set Cover Solution for Matrix 9:
Solution (g.s): [ True]

Set Cover Solution for Matrix 10:
Solution (g.s): [ True  True False]



**Converting everything into sequence:**

In [9]:
def convert_to_sequences(binary_matrices, solutions):
    sequences = []

    for binary_matrix, solution in zip(binary_matrices, solutions):
        # Convert binary matrix to a sequence of subsets
        input_sequence = [list(np.where(column)[0]) for column in binary_matrix.T]

        # Convert solution to a sequence of integers
        #label_sequence = list(np.where(solution)[0])
        # Convert solution to a sequence of integers starting from 1
        label_sequence = [index + 1 for index in np.where(solution)[0]]

        sequences.append((input_sequence, label_sequence))

    return sequences

In [10]:
# (Assuming 'generated_matrices' contains the binary matrices and 'setcover_solutions' contains the SetCoverPy solutions)
# Input is the list of matrices, output is the list of integers

# Convert binary matrices and solutions to sequences
sequences = convert_to_sequences(generated_matrices, setcover_solutions)

# Print the converted sequences
for i, (input_sequence, label_sequence) in enumerate(sequences):
    print(f"Sequence {i + 1} - Input:")
    print(input_sequence)
    print("Label:")
    print(label_sequence)
    print()

Sequence 1 - Input:
[[1, 4], [0, 4], [2, 5], [2, 3, 4], [4]]
Label:
[1, 2, 3, 4]

Sequence 2 - Input:
[[0, 2], [2], [1], [0], [2]]
Label:
[1, 3]

Sequence 3 - Input:
[[3], [1, 5], [0, 2], [1, 2, 3, 4]]
Label:
[2, 3, 4]

Sequence 4 - Input:
[[0], [0], [1, 2], [2], [2]]
Label:
[1, 3]

Sequence 5 - Input:
[[1, 3], [1], [2], [0, 1], [0]]
Label:
[1, 2, 3, 4]

Sequence 6 - Input:
[[2, 3], [2, 5], [0, 3], [1, 3, 4]]
Label:
[2, 3, 4]

Sequence 7 - Input:
[[0, 1, 2], [0], [0, 3], [2]]
Label:
[1, 3]

Sequence 8 - Input:
[[3, 4], [2, 3, 4], [3], [0, 1, 3]]
Label:
[2, 4]

Sequence 9 - Input:
[[0, 1, 2, 3, 4]]
Label:
[1]

Sequence 10 - Input:
[[1, 2, 3, 4], [0, 4], [3]]
Label:
[1, 2]

